# CTC Model Training Pipeline
This notebook implements the Connectionist Temporal Classification (CTC) pipeline. 
Unlike the sliding window approach, this trains the `CTC_CRNN` sequentially on entire audio recordings using PyTorch's native `CTCLoss`.

In [8]:
import sys

assert sys.version_info >= (3, 10)
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    !git clone https://github.com/stachuapa123/ASR_project.git
    %cd ASR_project
    # !git checkout <YOUR_BRANCH_NAME>  # Uncomment and set this to your branch if needed
    !pip install -q torchmetrics
    from google.colab import drive

    drive.mount("/content/drive")

    # Extract data securely if on Colab
    !mkdir -p "/content/asr_data"
    !unzip -q "/content/drive/MyDrive/asr_data.zip" -d "/content/asr_data"
    DATA_DIR = "/content/asr_data"
else:
    # Local path
    %load_ext autoreload
    %autoreload 2
    DATA_DIR = "../data/1-500"  # Update to your local subset or AutorskieDane

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import torch
from torch.utils.data import DataLoader

from src.ctc.config import CTCConfig as C
from src.ctc.model import CTCModel
from src.ctc.dataset import CTCDataset, ctc_collate_fn
from src.ctc.augmentation import SpecAugment
from src.ctc.training import train_ctc

In [10]:
hparams = {
    "batch_size": 16,
    "epochs": 5,
    "optimizer": "AdamW",
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "max_lr": 3e-3,
    "pct_start": 0.2,
    "num_workers": 4,
    "specaug_freq_mask": 0.2,
    "specaug_time_mask": 0.125,
    "specaug_p": 0.5,
}

device = C.get_device()
print(f"Using device: {device}")

Using device: cuda


In [11]:
dataset = CTCDataset(
    data_root=DATA_DIR,
    cache_mode=True,
    apply_augmentations=False,
)

n_total = len(dataset)
n_val = max(1, int(0.15 * n_total))
n_train = n_total - n_val
generator = torch.Generator().manual_seed(42)
train_set, val_set = torch.utils.data.random_split(
    dataset,
    [n_train, n_val],
    generator=generator,
)
print(f"Train items: {len(train_set)} | Val items: {len(val_set)}")

train_loader = DataLoader(
    train_set,
    batch_size=hparams["batch_size"],
    shuffle=True,
    collate_fn=ctc_collate_fn,
    num_workers=hparams["num_workers"],
    pin_memory=True,
)
val_loader = DataLoader(
    val_set,
    batch_size=hparams["batch_size"],
    shuffle=False,
    collate_fn=ctc_collate_fn,
    num_workers=hparams["num_workers"],
    pin_memory=True,
)

Pre-computing and caching 1471 files...


CTC cache: 100%|██████████| 1471/1471 [00:07<00:00, 202.39it/s]

Cache complete. Total items in RAM: 1471
Train items: 1251 | Val items: 220


In [12]:
model = CTCModel()
criterion = torch.nn.CTCLoss(blank=C.N_CLASSES, zero_infinity=True)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hparams["lr"],
    weight_decay=hparams["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=hparams["max_lr"],
    steps_per_epoch=len(train_loader),
    epochs=hparams["epochs"],
    pct_start=hparams["pct_start"],
)
scaler = torch.amp.GradScaler(
    device=device,
    enabled=(device.type == "cuda"),
)
spec_aug = SpecAugment(
    freq_mask_percent=hparams["specaug_freq_mask"],
    time_mask_percent=hparams["specaug_time_mask"],
    p=hparams["specaug_p"],
)

# reuse hparams as checkpoint config (optionally add DATA_DIR, etc.)
config = {
    **hparams,
    "data_root": DATA_DIR,
    "model": "CTCModel",
}

In [13]:
model = train_ctc(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    n_epochs=hparams["epochs"],
    spec_augment=spec_aug,
    scheduler=scheduler,
    scaler=scaler,
    save_best_to="../trained_models/ctc_test_model.pt",
    checkpoint_config=config,
)

Epoch   1/5 | Train Loss: 3.8035 | Val Loss: 3.3501 | LR: 3.0e-03 [BEST]          
Epoch   2/5 | Train Loss: 3.3029 | Val Loss: 3.2092 | LR: 2.6e-03 [BEST]          
Epoch   3/5 | Train Loss: 2.8470 | Val Loss: 2.4500 | LR: 1.5e-03 [BEST]          
Epoch   4/5 | Train Loss: 2.2067 | Val Loss: 1.9820 | LR: 4.3e-04 [BEST]          
Epoch   5/5 | Train Loss: 1.9769 | Val Loss: 1.9054 | LR: 8.6e-08 [BEST]          
Restored best weights with Val Loss = 1.9054
